# Point-in-time ideas, on synthetic data

This notebook needs no database and no internet. Every number below is **synthetic** and defined in the notebook.
It calls the project's own pure functions (`src/quality.py`, `src/reconcile.py`) where they exist, and mimics the SQL
objects (`price_observations`, `observations_as_of`, `universe_members`) with small pandas tables.

1. A provisional row becomes accepted
2. An "as of" query returns what the database knew at an earlier time
3. A Yahoo vs NSE disagreement is classified
4. Universe membership is dated

Run from the repository root or from `notebooks/`.

In [ ]:
import sys
from pathlib import Path
from datetime import date, datetime, timedelta, timezone
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import quality, reconcile

IST = timezone(timedelta(hours=5, minutes=30))
def ist(y, m, d, hh, mm=0):
    return datetime(y, m, d, hh, mm, tzinfo=IST)

## 1. A provisional row becomes accepted

At ingest a row is judged only on data up to itself. A normal row is stored as `provisional`. A later run that
re-fetches the **same** value at least a day after the session closed appends an `accepted` transition
(`reconfirmed_by_later_fetch`). The earlier state is never overwritten.

In [ ]:
# (price_date, close, adj_close, current_state) for a synthetic ticker DEMO.NS
series = [
    (date(2026, 10, 12), 100.00, None, "accepted"),
    (date(2026, 10, 13), 101.50, None, "accepted"),
    (date(2026, 10, 14), 102.10, None, "provisional"),
]

# Too early: the evening of the same session. Nothing changes.
early = quality.decide_transitions("DEMO.NS", series, now=ist(2026, 10, 14, 20), refetched={date(2026, 10, 14): 102.10})
print("same evening:", early)

# Next evening, the re-fetch returns the same close -> accepted.
later = quality.decide_transitions("DEMO.NS", series, now=ist(2026, 10, 15, 20), refetched={date(2026, 10, 14): 102.10})
pd.DataFrame([t.__dict__ for t in later])

In [ ]:
# The append-only state log for that stock-day
quality_states = pd.DataFrame([
    {"ticker": "DEMO.NS", "price_date": date(2026, 10, 14), "state": "provisional", "reason": "new_observation",
     "decided_at": ist(2026, 10, 14, 17, 10)},
    {"ticker": "DEMO.NS", "price_date": date(2026, 10, 14), "state": later[0].state, "reason": later[0].reason,
     "decided_at": ist(2026, 10, 15, 17, 10)},
])

def quality_as_of(ts):
    known = quality_states[quality_states.decided_at <= ts]
    return known.sort_values("decided_at").groupby(["ticker", "price_date"]).tail(1)

print(quality_as_of(ist(2026, 10, 14, 18))[["price_date", "state"]].to_string(index=False))
print(quality_as_of(ist(2026, 10, 16, 9))[["price_date", "state"]].to_string(index=False))

## 2. An "as of" query

`price_observations` is append-only: every fetch adds a row with the time it was retrieved. `observations_as_of(ts)`
returns the latest value **seen** for each stock-day by time `ts`. Here Yahoo revised the 14 Oct close on a later
fetch; asking as of the earlier time still returns the old value, so a backtest cannot see the revision early.

In [ ]:
price_observations = pd.DataFrame([
    # ticker,    price_date,         close,  retrieved_at
    ("DEMO.NS", date(2026, 10, 13), 101.50, ist(2026, 10, 13, 17, 10)),
    ("DEMO.NS", date(2026, 10, 14), 102.10, ist(2026, 10, 14, 17, 10)),
    ("DEMO.NS", date(2026, 10, 14), 102.35, ist(2026, 10, 16, 17, 10)),  # revised by the source later
    ("DEMO.NS", date(2026, 10, 15), 103.00, ist(2026, 10, 15, 17, 10)),
], columns=["ticker", "price_date", "close", "retrieved_at"])

def observations_as_of(ts):
    seen = price_observations[price_observations.retrieved_at <= ts]
    return seen.sort_values("retrieved_at").groupby(["ticker", "price_date"]).tail(1).sort_values("price_date")

print("As of 15 Oct 20:00 IST:")
print(observations_as_of(ist(2026, 10, 15, 20)).to_string(index=False))
print("\nAs of now (17 Oct):")
print(observations_as_of(ist(2026, 10, 17, 9)).to_string(index=False))

## 3. Yahoo vs NSE disagreement, classified

`reconcile.classify` compares the stored Yahoo close with the **unadjusted** official NSE bhavcopy close and names
the most likely reason for any difference. The bhavcopy rows below are synthetic.

In [ ]:
cases = [
    # label, yahoo_close, bhavcopy row (close, last, prev_close, low, high)
    ("agrees",                 250.00, dict(close=250.00, last=250.10, prev_close=248.00, low=247.0, high=252.0)),
    ("last traded, not close", 250.10, dict(close=250.00, last=250.10, prev_close=248.00, low=247.0, high=252.0)),
    ("stale previous close",   248.00, dict(close=250.00, last=250.10, prev_close=248.00, low=247.0, high=252.0)),
    ("split rescaled (1:2)",   125.00, dict(close=250.00, last=250.10, prev_close=248.00, low=247.0, high=252.0)),
    ("intraday snapshot",      251.40, dict(close=250.00, last=250.10, prev_close=248.00, low=247.0, high=252.0)),
    ("unexplained",            263.00, dict(close=250.00, last=250.10, prev_close=248.00, low=247.0, high=252.0)),
    ("missing in bhavcopy",    250.00, None),
]
rows = []
for label, y, bhav in cases:
    o = reconcile.classify(y, bhav)
    rows.append({"case": label, "yahoo": y, "nse_close": bhav and bhav["close"],
                 "class": o.classification, "diff_bp": None if o.diff_bp is None else round(o.diff_bp, 1),
                 "detail": o.detail})
pd.DataFrame(rows)

## 4. Dated universe membership

`universe_membership` stores who was in a universe and **when**, so a backtest on 2020-06-01 uses the members of
that day, not today's list (no survivorship bias). The table below is synthetic; `universe_members(u, d)` is
mimicked with a half-open interval `[valid_from, valid_to)`.

In [ ]:
universe_membership = pd.DataFrame([
    ("dashboard", "ALPHA", date(2015, 1, 1), None,              "founding member"),
    ("dashboard", "BETA",  date(2015, 1, 1), date(2022, 3, 31), "replaced: demerger"),
    ("dashboard", "GAMMA", date(2022, 3, 31), None,             "replacement for BETA"),
    ("dashboard", "DELTA", date(2018, 7, 1), date(2021, 1, 1),  "dropped: delisted"),
], columns=["universe", "symbol", "valid_from", "valid_to", "reason"])

def universe_members(u, d):
    m = universe_membership
    live = (m.universe == u) & (m.valid_from <= d) & (m.valid_to.isna() | (m.valid_to.map(lambda v: v is not None and d < v)))
    return sorted(m[live].symbol)

for d in [date(2016, 1, 1), date(2020, 6, 1), date(2022, 3, 30), date(2022, 3, 31), date(2026, 11, 5)]:
    print(d, universe_members("dashboard", d))

## Takeaways

- Quality is a **timeline**, not a flag: `provisional → accepted` is appended when later evidence arrives.
- An **as-of** query answers "what did we know then", which keeps revisions out of the past.
- A Yahoo/NSE difference gets a **named cause**, not just a tolerance breach.
- Membership has **dates**, so historical tests use historical universes.

See `docs/DATA_LINEAGE.md` for the database versions of each object.